In [6]:
from pathlib import Path
import sys

import chromadb

repo_root = next(
    (path for path in [Path.cwd(), *Path.cwd().resolve().parents] if (path / "pyproject.toml").exists()),
    Path.cwd().resolve(),
)
sys.path.insert(0, str(repo_root))

from app.config import CHROMA_HOST, CHROMA_PORT, CHROMA_SSL, COLLECTION_NAME
from app.factory import get_embeddings

client = chromadb.HttpClient(host=CHROMA_HOST, port=CHROMA_PORT, ssl=CHROMA_SSL)
collection = client.get_collection(COLLECTION_NAME)
embeddings = get_embeddings()

/home/mahee/Work/Thesis/Repos/langchain-rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
# Peek at stored documents
peek_results = collection.peek(5)
peek_results

{'ids': ['5886c29bac3b614f4123c1bbfdf67322',
  '47ab7c9ac25ec4eb3226a5fb6184a14c',
  'c0900351ec45209816e678eac20b7efa',
  '8dbf26a6a2f82a0a6700a024866013cd',
  '89ca1d52015bdadadb50a164dffffb9d'],
 'embeddings': array([[-0.01252601,  0.00430641, -0.17723158, ..., -0.03023094,
         -0.0445499 , -0.0214947 ],
        [-0.01825344,  0.10490812, -0.15696892, ..., -0.05568815,
         -0.02141853, -0.00144539],
        [-0.0014521 ,  0.07073161, -0.15685971, ..., -0.07342072,
         -0.00941774,  0.0155003 ],
        [ 0.01129713,  0.06804923, -0.17887177, ..., -0.07536747,
         -0.02908905, -0.01136832],
        [ 0.00879104,  0.06498616, -0.172338  , ..., -0.0557088 ,
         -0.04451359,  0.00106392]], shape=(5, 768)),
 'metadatas': [{'title': 'README',
   'section': '',
   'description': '',
   'format': 'md',
   'content_type': 'narrative',
   'source_corpus': 'unknown',
   'source_file': 'knowledge_ingestion/content/v2/README.md'},
  {'source_file': 'knowledge_ingestion/c

In [12]:
# Query exactly like your RAG retriever does — see what it returns
query = "What is NISQ mean"
query_embedding = embeddings.embed_query(query)
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5,
    include=["documents", "metadatas", "distances"],
)

for doc, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
    print(f"Distance: {dist:.4f} | Source: {meta}")
    print(doc[:300])
    print("---")

Distance: 0.5552 | Source: {'format': 'mdx', 'source_file': 'knowledge_ingestion/content/v2/tech_docs/mlflow/docs/docs/classic-ml/deployment/deploy-model-locally/index.mdx', 'h1': 'Deploy MLflow Model as a Local Inference Server', 'source_corpus': 'mlflow', 'content_type': 'narrative', 'section': 'Deploy MLflow Model as a Local Inference Server > Troubleshooting', 'description': '', 'title': 'index', 'h2': 'Troubleshooting'}
## Troubleshooting
---
Distance: 0.5714 | Source: {'title': 'index', 'h2': 'Get Started', 'format': 'mdx', 'description': '', 'h1': 'MLflow for Deep Learning', 'source_corpus': 'mlflow', 'source_file': 'knowledge_ingestion/content/v2/tech_docs/mlflow/docs/docs/classic-ml/deep-learning/index.mdx', 'section': 'MLflow for Deep Learning > Get Started', 'content_type': 'narrative'}
## Get Started
---
Distance: 0.5714 | Source: {'source_corpus': 'mlflow', 'content_type': 'narrative', 'h2': 'Learn More', 'title': 'index', 'description': '', 'section': 'MLflow Sentence Tra

In [13]:
results

{'ids': [['f253a5d2a95b0c29a13329b8eeb8db41',
   '2632b97e80c86302999ab55189ee0222',
   '042ab836bcf7dcf1a3f63759b268d4d5',
   'd8f33811e8a1c1ab56c903be5c1d66ef',
   'c8f75b972115c87a15e16d9421e573e1']],
 'distances': [[0.5551642, 0.57136405, 0.57136405, 0.59431684, 0.5971663]],
 'embeddings': None,
 'metadatas': [[{'format': 'mdx',
    'source_file': 'knowledge_ingestion/content/v2/tech_docs/mlflow/docs/docs/classic-ml/deployment/deploy-model-locally/index.mdx',
    'h1': 'Deploy MLflow Model as a Local Inference Server',
    'source_corpus': 'mlflow',
    'content_type': 'narrative',
    'section': 'Deploy MLflow Model as a Local Inference Server > Troubleshooting',
    'description': '',
    'title': 'index',
    'h2': 'Troubleshooting'},
   {'title': 'index',
    'h2': 'Get Started',
    'format': 'mdx',
    'description': '',
    'h1': 'MLflow for Deep Learning',
    'source_corpus': 'mlflow',
    'source_file': 'knowledge_ingestion/content/v2/tech_docs/mlflow/docs/docs/classic-ml